In [1]:
# ==============================
# CELL 1: IMPORT LIBRARIES
# ==============================

# HuggingFace dataset loader
from datasets import load_dataset

# Transformer tools (tokenizer + model + trainer)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# Loss function for multi-label classification
from torch.nn import BCEWithLogitsLoss

# Utilities
import numpy as np
import torch

In [2]:
# ==============================
# CELL 2: LOAD DATASET
# ==============================

# Load GoEmotions dataset (multi-label emotion dataset)
dataset = load_dataset("go_emotions")

# 🔥 SPEED OPTIMIZATION:
# Use smaller subset to reduce training time (very important for laptops)
dataset["train"] = dataset["train"].select(range(5000))
dataset["validation"] = dataset["validation"].select(range(1000))

# ✅ This still keeps your project valid

In [3]:
# ==============================
# CELL 3: LOAD TOKENIZER
# ==============================

# Convert text into tokens that BERT understands
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [4]:
# ==============================
# CELL 4: PREPROCESSING
# ==============================

# Total number of emotion labels in GoEmotions
num_labels = 28

def preprocess(example):
    # Convert text → token IDs
    encoding = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64   # 🔥 smaller length = faster training
    )
    
    # Convert labels → multi-hot vector
    # Example: [0,3] → [1,0,0,1,0,...]
    multi_hot = np.zeros(num_labels, dtype=np.float32)
    
    for label in example["labels"]:
        multi_hot[label] = 1.0
    
    encoding["labels"] = multi_hot
    return encoding

# Apply preprocessing to dataset
encoded_dataset = dataset.map(preprocess)

# Convert to PyTorch tensors
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ✅ Fixes:
# - multi-label problem
# - dtype issues
# - Trainer compatibility

In [5]:
# ==============================
# CELL 5: LOAD MODEL
# ==============================

# Load BERT for classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"  # 🔥 important
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# ==============================
# CELL 6: CUSTOM TRAINER
# ==============================

# Why custom trainer?
# HuggingFace sometimes mishandles multi-label loss (dtype issues)
# So we override it manually

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        
        # Force labels to float (fixes all errors)
        labels = inputs.pop("labels").float()
        
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Multi-label loss
        loss_fn = BCEWithLogitsLoss()
        loss = loss_fn(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [7]:
# ==============================
# CELL 7: TRAINING ARGUMENTS
# ==============================

training_args = TrainingArguments(
    output_dir="./results",
    
    # Evaluate + save after each epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    
    # 🔥 MEMORY SAFE (Mac fix)
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    
    # 🔥 Simulates bigger batch without memory crash
    gradient_accumulation_steps=4,
    
    # Keep small for project
    num_train_epochs=3,
    
    logging_steps=50
)

In [8]:

#  INITIALIZE TRAINER

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
)

In [9]:

# TRAIN MODEL

trainer.train()

# 🎯 Expected output:
# loss: 0.xxx

# ✅ This means your model is learning emotions
trainer.save_model("./emotion_model")
tokenizer.save_pretrained("./emotion_model")

/Users/apple/chatbot_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.621749,0.157666
2,0.588165,0.157264
3,0.587671,0.156811


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/apple/chatbot_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/apple/chatbot_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./emotion_model/tokenizer_config.json', './emotion_model/tokenizer.json')